In [ ]:
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch import Tensor
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import StepLR

In [ ]:
from typing import Sequence, Union

def get_activation(name: str):
    """
    Return a torch activation callable from a string name. Supported:
    "relu", "tanh", "sigmoid", "leaky_relu", "elu".
    """
    _activations = {
        "relu":       torch.relu,
        "tanh":       torch.tanh,
        "sigmoid":    torch.sigmoid,
        "leaky_relu": torch.nn.functional.leaky_relu,
        "elu":        torch.nn.functional.elu,
    }
    if name not in _activations:
        raise ValueError(
            f"Unknown activation '{name}'. Choose from {list(_activations)}"
        )
    return _activations[name]


class CNN(nn.Module):
    def __init__(
        self,
        activation: Union[str, callable] = "tanh",
    ):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1)
        # after global average pooling we have 64 features
        self.fc = nn.Linear(64, 10)
        
        if isinstance(activation, str):
                activation = get_activation(activation)
        self.activation = activation

    def forward(self, x):
        x = self.conv1(x)
        x = self.activation(x)
        x = self.conv2(x)
        x = self.activation(x)
        x = F.max_pool2d(x, 2)
        # (batch, 64, 12, 12) -> (batch, 64, 1, 1)
        x = F.adaptive_avg_pool2d(x, 1)
        # (batch, 64)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x
    

class FFN(nn.Module):
    """
    Plain fully-connected network. Stored as nn.ModuleList so that
    named_parameters() yields layers.0.weight, layers.0.bias, layers.1.weight,
    ..., matching ParamSpec.from_layer_sizes ordering.
    """
    def __init__(
        self,
        layer_sizes: Sequence[int],
        activation: Union[str, callable] = "tanh",
    ):
        super().__init__()

        if isinstance(activation, str):
            activation = get_activation(activation)

        self.layers = nn.ModuleList([
            nn.Linear(n_in, n_out)
            for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:])
        ])

        self.activation = activation
        self._n_layers = len(self.layers)

    def forward(self, x: Tensor) -> Tensor:
        h = x
        for i, layer in enumerate(self.layers):
            h = layer(h)
            if i < self._n_layers - 1:
                h = self.activation(h)
        return h

# transform=transforms.Compose([
#         transforms.ToTensor(),
#         transforms.Normalize((0.1307,), (0.3081,))
#         ])
# dataset1 = datasets.MNIST('../data', train=True, download=True,
#                        transform=transform)


In [ ]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
from sazz.utils.bnn_modular_utils import (
    ParamSpec,
    build_prior_precision,
    make_kappa_from_inclusion,
    build_ffn_module,
)
from sazz.models.bnn_torch import ModuleCategoricalLikelihood, ModuleGaussianPrior
from sazz.models.models_torch import BayesianModel

dtype=torch.float64
module = CNN().to(dtype=dtype)
spec = ParamSpec.from_module(module)
print(spec)

N = 128 
X_train = torch.randn(N, 1, 28, 28, dtype=torch.float64)
y_train = torch.randint(0, 10, (N,), dtype=torch.long)

prec = build_prior_precision(
    spec, prior_std_weight=1.0, prior_std_bias=1.0,
    fan_in_scaling=True, dtype=torch.float64, device="cpu",
)
prior = ModuleGaussianPrior(prec)
likelihood = ModuleCategoricalLikelihood(module, spec, X_train, y_train)
model = BayesianModel(prior, likelihood)

In [ ]:
from sazz.utils.warmup import find_reference_bnn
x_ref, Sigma_inv = find_reference_bnn(
        model.energy, spec.D, model=model, dtype=dtype, device="cpu",
        reference="laplace_diag", n_steps=2000, lr=1e-2,
    )

In [ ]:
from sazz.models.make_models import TorchTarget

target = TorchTarget(
    name="conv_mnist_bnn",
    D=spec.D,
    grad_target=model.grad_energy,
    x_ref=x_ref,
    Sigma_inv=Sigma_inv,
    meta={"model": model, "spec": spec, "module": module},
)

In [ ]:
sum(target.x_ref<=1e-5)/19466